In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OLLAMA_API_KEY"] = os.getenv("OLLAMA_API_KEY")

In [7]:
from langchain.chat_models import init_chat_model
model=init_chat_model("ollama:qwen3.5:4b")
model

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, model='qwen3.5:4b')

In [9]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [10]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={} response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-09-29T08:05:59.5923144Z', 'done': True, 'done_reason': 'stop', 'total_duration': 28780442200, 'load_duration': 20754744100, 'prompt_eval_count': 275, 'prompt_eval_duration': 1208217000, 'eval_count': 74, 'eval_duration': 6779459000, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'} id='lc_run--01a0ec32-8c39-7690-9147-95d5e5953bc0-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'e7bfb601-7faa-40ab-bb9a-5207e5368ed6', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 275, 'output_tokens': 74, 'total_tokens': 349}
Tool: get_weather
Args: {'location': 'Boston'}


In [12]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

It's sunny in Boston.
